# Graph Neural Networks (GNN)

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

A refresher on graph neural networks: how message passing turns a graph + node
features into useful node/edge/graph representations, with from-scratch NumPy
examples you can run on a laptop.


## 1. What & Why

A **graph neural network** learns over data shaped as a graph — nodes connected by
edges — where the *connections carry meaning* and there is no natural grid or
sequence order. Think social networks, molecules, knowledge graphs, road networks,
recommendation bipartite graphs, citation networks, and meshes.

CNNs exploit grid structure (fixed neighbourhoods, translation invariance) and RNNs
exploit sequence order. Neither fits a graph: nodes have *variable, unordered*
neighbourhoods, and the same node can sit in wildly different local topologies. A GNN
generalises the convolution idea to arbitrary graphs — each node updates its state by
**aggregating messages from its neighbours**, repeated for a few rounds so information
flows outward along edges.

**Reach for a GNN when** the relationships between samples are as informative as the
samples themselves, and you want to predict at the level of:
- **nodes** — classify users, label pixels in a superpixel graph, predict protein function;
- **edges** — link prediction / recommendation ("will these two connect?");
- **whole graphs** — molecule property prediction, predicting a circuit's behaviour.

**Skip it when** your data is already a clean grid (use a CNN), a sequence (use a
Transformer/RNN), or plain tabular rows with no relational structure (gradient-boosted
trees usually win). A GNN's value comes entirely from the edges — no meaningful edges,
no reason to pay its cost.


## 2. Mental Model

**A GNN is repeated, learnable "gossip" on a graph.** In each round every node:

1. **collects** a message from each neighbour (often just the neighbour's current vector),
2. **aggregates** those messages with a *permutation-invariant* function (sum / mean / max —
   order must not matter, because neighbours are an unordered set),
3. **updates** its own vector from the aggregate (a small MLP + nonlinearity).

```
        round k                       round k+1
   (b)                           every node = f( its old state,
     \                                            aggregate of neighbours' states )
(a)—(c)—(d)        ───────▶      so after K rounds each node has "heard"
     /                           everything within K hops of it.
   (e)
```

After **K** message-passing layers, a node's representation summarises its **K-hop
neighbourhood**. One layer = direct neighbours; two layers = neighbours-of-neighbours;
and so on. The whole network is just this aggregate-and-update step stacked K times, with
learned weights shared across all nodes (the analogue of a CNN's shared filter).

The classic **GCN** layer makes this concrete: `H' = σ(Â H W)`, where `Â` is the
normalised adjacency (with self-loops) that does the neighbour-averaging, `W` is the
learned per-feature transform, and `σ` is a nonlinearity.


## 3. Key Concepts

- **Message passing (MPNN)** — the unifying framework: `message → aggregate → update`,
  repeated per layer. Almost every GNN variant is a special case.
- **Permutation invariance / equivariance** — aggregation over neighbours must be
  order-independent (sum/mean/max, *not* concatenation). Node outputs are *equivariant*:
  relabel the nodes and the outputs relabel the same way.
- **Adjacency matrix `A`** — who connects to whom. GCN uses the **symmetric normalised**
  form `Â = D̃^(−1/2) (A+I) D̃^(−1/2)`: `+I` adds self-loops so a node keeps its own signal,
  and the `D̃^(−1/2)` factors stop high-degree nodes from blowing up the scale.
- **GCN** (Kipf & Welling) — fixed degree-based averaging of neighbours.
- **GraphSAGE** — sample a fixed number of neighbours and aggregate (mean/LSTM/pool);
  enables **inductive** learning on unseen nodes and big graphs.
- **GAT** (Graph Attention) — learn *attention weights* over neighbours instead of using
  fixed normalisation; lets the model decide which neighbours matter.
- **Readout / pooling** — to predict on a *whole graph*, pool all node vectors into one
  (sum/mean/max, or hierarchical pooling). Sum-pooling is the most expressive.
- **Receptive field = depth (K)** — K layers ⇒ K-hop receptive field. Unlike CNNs, going
  deep is *hard* here (see oversmoothing).
- **Transductive vs inductive** — transductive trains and predicts on one fixed graph
  (GCN's default); inductive generalises to new nodes/graphs (GraphSAGE, GAT).
- **WL test** — message-passing GNNs are at most as powerful as the 1-Weisfeiler-Lehman
  graph-isomorphism test at distinguishing graphs; **GIN** is designed to hit that bound.


## 4. Setup

The worked examples below use only **NumPy** (for the math) and **NetworkX** (to build
graphs and supply the classic Karate Club dataset) — both pure-CPU and tiny. No GPU, no
deep-learning framework needed to *understand* the mechanics.

For real work you would reach for a dedicated library:

- **PyTorch Geometric (PyG)** — `pip install torch torch_geometric` — the most popular;
  rich layer zoo, mini-batching of graphs, neighbour sampling.
- **Deep Graph Library (DGL)** — `pip install dgl` — backend-agnostic, scales well.

We deliberately implement message passing by hand so the abstraction is transparent.


In [1]:
# Pure-CPU deps for the from-scratch examples.
# %pip install numpy networkx
import numpy as np
import networkx as nx

print("numpy", np.__version__, "| networkx", nx.__version__)


numpy 2.4.6 | networkx 3.6.1


## 5. Worked Examples

### Example 1 — One GCN layer = normalised neighbour averaging

We build a tiny 5-node graph, form the symmetric-normalised adjacency `Â`, and apply a
single GCN layer `H' = σ(Â X W)` by hand. The point: a layer *smooths each node toward its
neighbours*, then linearly transforms the result.


In [2]:
def normalized_adjacency(G):
    """Symmetric-normalised adjacency with self-loops:  D̃^(-1/2)(A+I)D̃^(-1/2)."""
    A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
    A_tilde = A + np.eye(A.shape[0])          # add self-loops
    deg = A_tilde.sum(axis=1)                  # degrees (incl. self-loop)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(deg))
    return d_inv_sqrt @ A_tilde @ d_inv_sqrt

def relu(x):
    return np.maximum(x, 0.0)

# A small graph: a triangle (0-1-2) with a tail 2-3-4.
G = nx.Graph([(0, 1), (1, 2), (0, 2), (2, 3), (3, 4)])
A_hat = normalized_adjacency(G)

# 2 features per node; node 0 is "hot" to watch it diffuse outward.
X = np.zeros((5, 2))
X[0] = [1.0, 0.0]
X[4] = [0.0, 1.0]

rng = np.random.default_rng(0)
W = rng.normal(0, 0.5, size=(2, 3))           # learned transform: 2 -> 3 dims

H = relu(A_hat @ X @ W)                        # one GCN layer
np.set_printoptions(precision=3, suppress=True)
print("Â (rounded):\n", np.round(A_hat, 2))
print("\nNode embeddings after 1 GCN layer (5 nodes x 3 dims):\n", H)


Â (rounded):
 [[0.33 0.33 0.29 0.   0.  ]
 [0.33 0.33 0.29 0.   0.  ]
 [0.29 0.29 0.25 0.29 0.  ]
 [0.   0.   0.29 0.33 0.41]
 [0.   0.   0.   0.41 0.5 ]]

Node embeddings after 1 GCN layer (5 nodes x 3 dims):
 [[0.021 0.    0.107]
 [0.021 0.    0.107]
 [0.018 0.    0.092]
 [0.021 0.    0.074]
 [0.026 0.    0.09 ]]


Node 0's signal has spread to its neighbours 1 and 2 (and node 4's signal reached node 3)
— exactly one hop. Stack another layer and node 0 would influence node 3 as well. That
outward diffusion *is* the receptive field growing with depth.


### Example 2 — Semi-supervised node classification (the Karate Club)

Zachary's Karate Club split into two factions. We train a **2-layer GCN from scratch**
(forward + manual backprop) to predict each member's faction, using **only two labelled
nodes** — the two leaders (node 0 and node 33). The graph structure propagates those two
labels to everyone else. This is the canonical demo that made GCNs famous.


In [3]:
G = nx.karate_club_graph()
n = G.number_of_nodes()
A_hat = normalized_adjacency(G)

# Features: identity matrix (no node features -> let the model learn an embedding per node).
X = np.eye(n)

# True faction labels; we only let the model SEE two of them during training.
labels = np.array([0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)])
Y = np.eye(2)[labels]
train_mask = np.zeros(n, dtype=bool)
train_mask[[0, 33]] = True                     # only the two leaders are labelled

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

rng = np.random.default_rng(0)
hidden = 8
W0 = rng.normal(0, 0.5, size=(n, hidden))      # layer 1: n -> hidden
W1 = rng.normal(0, 0.5, size=(hidden, 2))      # layer 2: hidden -> 2 classes
lr = 0.5

for epoch in range(101):
    # ---- forward: H1 = relu(Â X W0);  Z = Â H1 W1 ----
    pre1 = A_hat @ X @ W0
    H1 = relu(pre1)
    Z = A_hat @ H1 @ W1
    P = softmax(Z)

    # ---- backward (cross-entropy on labelled nodes only) ----
    dZ = P.copy()
    dZ[train_mask] -= Y[train_mask]
    dZ[~train_mask] = 0.0
    dZ /= train_mask.sum()
    dW1 = (A_hat @ H1).T @ dZ
    dH1 = A_hat @ dZ @ W1.T
    dpre1 = dH1 * (pre1 > 0)
    dW0 = (A_hat @ X).T @ dpre1
    W0 -= lr * dW0
    W1 -= lr * dW1

    if epoch % 25 == 0:
        acc = (P.argmax(1) == labels).mean()
        print(f"epoch {epoch:3d}  full-graph accuracy {acc:.2f}")

print("\nFinal accuracy on all 34 members (trained on 2 labels):",
      f"{(softmax(A_hat @ relu(A_hat @ X @ W0) @ W1).argmax(1) == labels).mean():.2f}")


epoch   0  full-graph accuracy 0.50
epoch  25  full-graph accuracy 0.94
epoch  50  full-graph accuracy 0.94
epoch  75  full-graph accuracy 0.97
epoch 100  full-graph accuracy 0.97

Final accuracy on all 34 members (trained on 2 labels): 0.97


From **two** labelled nodes the GCN recovers almost the entire faction split — because
message passing carries label information along the edges. That is the whole promise of
semi-supervised learning on graphs: structure substitutes for labels.


## 6. Gotchas & Pitfalls

- **Oversmoothing.** Too many layers makes every node's representation converge to the same
  value (repeated averaging is a low-pass filter). In practice GCNs peak at **2–4 layers**.
  Mitigations: residual/skip connections, jumping-knowledge, PairNorm, or attention.
- **Over-squashing.** Information from an exponentially growing K-hop neighbourhood is
  crushed into a fixed-size vector, so long-range dependencies get lost — bottlenecked by
  graph topology. Rewiring or adding virtual/global nodes helps.
- **Limited expressivity (1-WL bound).** Standard message-passing GNNs cannot distinguish
  certain non-isomorphic graphs (e.g. some regular graphs). Use **GIN**-style sum
  aggregation, positional/structural encodings, or higher-order GNNs if this bites.
- **Scalability / neighbour explosion.** Full-batch GCN needs the whole adjacency in memory;
  full neighbourhoods blow up exponentially with depth. Use **GraphSAGE neighbour sampling**,
  Cluster-GCN, or GraphSAINT for large graphs.
- **Wrong normalisation.** Forgetting self-loops (`+I`) or symmetric normalisation makes
  training unstable or lets high-degree hubs dominate. The `Â` form exists for a reason.
- **Transductive leakage.** Plain GCN sees the *entire* graph (including test nodes) during
  message passing. That is fine transductively, but don't claim inductive generalisation —
  evaluate on a genuinely held-out graph for that.
- **Aggregation must be permutation-invariant.** Never concatenate neighbours in a fixed
  order; the model would depend on arbitrary node IDs. Use sum/mean/max.
- **Class/edge imbalance & isolated nodes.** Isolated nodes only see themselves (self-loop);
  highly imbalanced classes need weighting just like in any classifier.


## 7. When to Use vs Alternatives

| Situation | Better choice | Why |
|---|---|---|
| Data is genuinely relational (molecules, social/citation graphs, knowledge graphs) | **GNN** | Edges carry signal; message passing exploits them |
| Need fixed-size neighbours / scale to huge or streaming graphs / unseen nodes | **GraphSAGE** | Inductive, neighbour sampling, bounded compute |
| Some neighbours matter far more than others | **GAT** | Learns attention over edges instead of fixed weights |
| Whole-graph property prediction, max expressivity | **GIN + sum readout** | Hits the 1-WL bound; best graph-level discrimination |
| Long-range dependencies, dense interactions | **Graph Transformer** | Global attention sidesteps over-squashing (at O(n²) cost) |
| Plain tabular rows, no real edges | **Gradient-boosted trees / MLP** | A GNN's edges would be noise; trees usually win |
| Grid data (images) or sequences (text/audio) | **CNN / Transformer** | Those inductive biases fit better than a generic graph |
| You *can* fabricate a graph (e.g. kNN over tabular features) | **Try the simpler model first** | An invented graph often adds cost without signal |

**Variant cheat-sheet:** GCN (simple, transductive, fixed averaging) → GraphSAGE (inductive,
sampling) → GAT (attention weights) → GIN (max expressivity, graph-level) → Graph
Transformer (global attention, long range). Start with GCN/GraphSAGE; reach for the rest
only when a specific limitation bites.


## 8. Resources

- **Kipf & Welling (2017), *Semi-Supervised Classification with GCNs*** — the GCN paper:
  https://arxiv.org/abs/1609.02907
- **Hamilton, Ying & Leskovec (2017), *GraphSAGE*** — inductive representation learning:
  https://arxiv.org/abs/1706.02216
- **Veličković et al. (2018), *Graph Attention Networks*** — https://arxiv.org/abs/1710.10903
- **Gilmer et al. (2017), *Neural Message Passing for Quantum Chemistry*** — the MPNN
  framework that unifies these models: https://arxiv.org/abs/1704.01212
- **Distill, *A Gentle Introduction to Graph Neural Networks*** — the best visual primer:
  https://distill.pub/2021/gnn-intro/
- **PyTorch Geometric docs** — the practical library to use in production:
  https://pytorch-geometric.readthedocs.io/
- **Stanford CS224W, *Machine Learning with Graphs*** — full course:
  https://web.stanford.edu/class/cs224w/
